In [ ]:
# pip install geopandas shapely matplotlib
import geopandas as gpd
import matplotlib.pyplot as plt
import unicodedata
from pathlib import Path

# === 1) مسیرها را تنظیم کن (اینجا را به فولدرهای خودت تغییر بده) ===
# مثال‌های رایج:
# r"D:\data\natural_earth_vector\10m_cultural\ne_10m_admin_1_states_provinces.shp"
# r"D:\data\ne_10m_admin_0_countries\ne_10m_admin_0_countries.shp"
ADMIN1_PATH = r"D:\projects\climate\natural_earth_vector\10m_cultural\ne_10m_admin_1_states_provinces.shp"
ADMIN0_PATH = r"D:\projects\climate\ne_10m_admin_0_countries\ne_10m_admin_0_countries.shp"

assert Path(ADMIN1_PATH).exists(), "مسیر Admin-1 (استان‌ها) درست نیست."
assert Path(ADMIN0_PATH).exists(), "مسیر Admin-0 (کشورها) درست نیست."

# === 2) داده‌ها را بخوان ===
provinces = gpd.read_file(ADMIN1_PATH).to_crs(epsg=4326)
countries = gpd.read_file(ADMIN0_PATH).to_crs(epsg=4326)

# ستون‌های نام در NE معمولاً این‌ها هستند:
#  - 'name' یا 'name_en' یا 'name_local' برای استان‌ها
#  - برای کشورها 'ADMIN' یا 'SOVEREIGNT' یا 'NAME'
# بیابیم کدام ستون‌ها وجود دارد:
prov_name_col = next(c for c in provinces.columns if c.lower() in ["name", "name_en", "name_local", "gn_name"])
ctry_name_col = next(c for c in countries.columns if c.lower() in ["admin", "name", "name_long", "sovereignt", "geounit"])

# === 3) فیلتر کشور ایران ===
iran = countries[countries[ctry_name_col].str.contains("Iran", case=False, na=False)]

if iran.empty:
    raise ValueError("کشور ایران در لایه Admin-0 پیدا نشد. ستون نام را چک کن.")

# === 4) پیدا کردن استان خوزستان (robust matching) ===
def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return s.lower()

alt_names = ["khuzestan", "khuzistan", "khuzestan province"]  # پوشش واریانت‌ها
mask_iran_provs = provinces.with_columns if hasattr(provinces, "with_columns") else None  # سازگاری نسخه‌ها

# برخی shapefileها ستون کشور/iso دارند:
country_cols = [c for c in provinces.columns if c.lower() in ["adm0_a3", "sr_adm0_a3", "iso_a3", "gn_level", "geonunit", "admin"]]
if country_cols:
    # سعی کن فقط استان‌های داخل ایران را نگه داری
    # Natural Earth معمولاً ADM0_A3 = "IRN" دارد
    ccol = country_cols[0]
    provs_in_iran = provinces[provinces[ccol].astype(str).str.contains("IR", case=False, na=False)]
    if provs_in_iran.empty:
        provs_in_iran = provinces  # اگر ستون کاربردی نبود، روی همه بگرد
else:
    provs_in_iran = provinces

# جست‌وجوی نام خوزستان
khz = provs_in_iran[provs_in_iran[prov_name_col].apply(lambda x: any(a in norm(str(x)) for a in alt_names))]

if khz.empty:
    # اگر پیدا نشد، لیست چند استان برای کمک چاپ کن
    sample = provs_in_iran[prov_name_col].head(30).tolist()
    raise ValueError(f"خوزستان پیدا نشد. چند نام موجود: {sample[:10]} ...  ستون نام: {prov_name_col}")

# === 5) ترسیم ===
fig, ax = plt.subplots(figsize=(8, 9))

# مرز ایران
iran.boundary.plot(ax=ax, linewidth=1.2)
iran.plot(ax=ax, alpha=0.05)

# همه استان‌های ایران (کم‌رنگ)
provs_in_iran.plot(ax=ax, alpha=0.03, edgecolor="black", linewidth=0.3)

# هایلایت خوزستان
khz.boundary.plot(ax=ax, linewidth=2)
khz.plot(ax=ax, alpha=0.15, hatch="//", edgecolor=None)

# حد دید مناسب (روی مرز ایران زوم می‌کنیم؛ کمی حاشیه)
minx, miny, maxx, maxy = iran.total_bounds
dx, dy = (maxx - minx) * 0.05, (maxy - miny) * 0.05
ax.set_xlim(minx - dx, maxx + dx)
ax.set_ylim(miny - dy, maxy + dy)

#ax.set_title("Iran with Khuzestan Highlighted", fontsize=12)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

# برچسب داخل استان
try:
    cx, cy = khz.geometry.unary_union.representative_point().x, khz.geometry.unary_union.representative_point().y
    ax.text(cx, cy, "Khuzestan", fontsize=10, ha="center", va="center")
except Exception:
    pass

plt.tight_layout()
plt.savefig("iran_khuzestan_highlight.png", dpi=1000)
plt.show()

# اگر خواستی خروجی را ذخیره کن:


In [ ]:
# -*- coding: utf-8 -*-
# Trend lines: decadal average of annual hazard occurrences (1940–2024)
# pip install matplotlib pandas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

# ===================== USER PATHS =====================
OUT_DIR = Path(r"D:\projects\climate\exports")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PNG_PATH = OUT_DIR / "hazard_trends_decadal_FALL_SPRING.png"
PDF_PATH = OUT_DIR / "hazard_trends_decadal_FALL_SPRING.pdf"

SEASON_FILES = {
    "FALL": {
        "NW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedNW.csv",
        "NE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedNE.csv",
        "SW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedSW.csv",
        "SE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedSE.csv",
    },
    "SPRING": {
        "NW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedNW.csv",
        "NE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedNE.csv",
        "SW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedSW.csv",
        "SE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedSE.csv",
    }
}

# ===================== CONSTANTS =====================
HAZ_BIN = [
    "hazard_drought",
    "hazard_rain_deficit",
    "hazard_heatwave",
    "hazard_hot_and_dry",
    "hazard_cold_spell",
    "hazard_radiation_deficit",
    "hazard_temperature_accumulation_deficit",
]
HAZ_LABELS_EN = {
    "hazard_drought": "Drought",
    "hazard_rain_deficit": "Rain Deficit",
    "hazard_heatwave": "Heatwave",
    "hazard_hot_and_dry": "Hot & Dry (Compound)",
    "hazard_cold_spell": "Cold Spell",
    "hazard_radiation_deficit": "Radiation Deficit",
    "hazard_temperature_accumulation_deficit": "Temperature Accumulation Deficit",
}
# رنگ‌ها با کنتراست مناسب
HAZ_COLORS = {
    "hazard_drought": "#e41a1c",
    "hazard_rain_deficit": "#377eb8",
    "hazard_heatwave": "#ff7f00",
    "hazard_hot_and_dry": "#a65628",
    "hazard_cold_spell": "#984ea3",
    "hazard_radiation_deficit": "#dede00",
    "hazard_temperature_accumulation_deficit": "#4daf4a",
}
# اگر فقط *_prob داری، حضور را با این آستانه تعریف کن:
PROB_THRESHOLD = 0.5

# ===================== HELPERS =====================
def read_table_any(path: str) -> pd.DataFrame:
    p = Path(path)
    suf = p.suffix.lower()
    if suf in [".csv", ".txt"]:   return pd.read_csv(p)
    if suf == ".parquet":         return pd.read_parquet(p)
    if suf in [".xlsx", ".xls"]:  return pd.read_excel(p)
    return pd.read_csv(p)

def compute_presence_matrix(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    for base in HAZ_BIN:
        bin_col  = base if base in df.columns else None
        prob_col = base + "_prob" if base + "_prob" in df.columns else None
        bin_vals  = df[bin_col].astype(float) if bin_col else 0.0
        prob_vals = (df[prob_col] >= PROB_THRESHOLD).astype(float) if prob_col else 0.0
        out[base] = ((pd.Series(bin_vals, index=df.index).fillna(0) +
                      pd.Series(prob_vals, index=df.index).fillna(0)) > 0).astype(int)
    return out

def load_season(files_dict: dict) -> pd.DataFrame:
    """داده‌های ۴ ناحیه را یکجا برمی‌گرداند؛ year باید موجود باشد."""
    frames = []
    for _, p in files_dict.items():
        df = read_table_any(p)
        if "year" not in df.columns:
            raise KeyError(f"'year' column not found in {p}")
        pres = compute_presence_matrix(df)
        for col in HAZ_BIN:
            if col not in df.columns:
                df[col] = pres[col]
            else:
                df[col] = df[col].fillna(pres[col]).astype(int)
        frames.append(df[["year", *HAZ_BIN]])
    return pd.concat(frames, ignore_index=True)



def decadal_mean_annual_counts(df: pd.DataFrame) -> pd.DataFrame:
    """
    1) برای هر سال تعداد رخداد هر هازارد در کل استان را می‌شماریم.
    2) دهه را از سال می‌سازیم (1940s, 1950s, ...).
    3) میانگین سالانه در هر دهه (برای قابل‌مقایسه بودن دهه‌ی ناقص 2020s).
    خروجی: index=دهه (label مانند '1940s'), columns=HAZ_BIN
    """
    # شمارش سالانه
    annual = df.groupby("year")[HAZ_BIN].sum()
    # برچسب دهه
    decade = (annual.index // 10) * 10
    annual["decade"] = decade
    # میانگین سالانه در هر دهه
    decadal = annual.groupby("decade")[HAZ_BIN].mean()
    # برچسب‌های زیباتر
    decadal.index = [f"{int(d)}s" for d in decadal.index]
    return decadal

# ===================== LOAD & AGG =====================
fall_df   = load_season(SEASON_FILES["FALL"])
spring_df = load_season(SEASON_FILES["SPRING"])

fall_decadal   = decadal_mean_annual_counts(fall_df)
spring_decadal = decadal_mean_annual_counts(spring_df)

# دهه‌های مشترک برای دو فصل (برای محور X یکسان)
all_decades = sorted(set(fall_decadal.index).union(set(spring_decadal.index)))
fall_decadal   = fall_decadal.reindex(all_decades)
spring_decadal = spring_decadal.reindex(all_decades)

# ===================== PLOT =====================
mpl.rcParams.update({
    "font.size": 8,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), dpi=450, sharey=True)
seasons = [("Autumn", fall_decadal), ("SPRING", spring_decadal)]

for ax, (season_name, dec_df) in zip(axes, seasons):
    # محور X = دهه‌ها
    x = np.arange(len(dec_df.index))
    for haz in HAZ_BIN:
        y = dec_df[haz].to_numpy()
        ax.plot(x, y, marker="o", linewidth=1.8,
                label=HAZ_LABELS_EN[haz], color=HAZ_COLORS[haz], alpha=0.95)
    ax.set_xticks(x, dec_df.index, rotation=0)
    ax.set_xlabel("Decade")
    ax.set_title(f"{season_name} — Decadal mean of annual occurrences")
    ax.grid(linestyle=":", alpha=0.5)
    ax.set_xlim(-0.2, len(x)-0.8)

axes[0].set_ylabel("Mean annual number of occurrences (per decade)")

# لجند مشترک
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3,
           title="Hazard Types", fontsize=8, title_fontsize=9)

#fig.suptitle("Khuzestan — Decadal hazard trends (1940–2024)\nLines show decadal mean of annual occurrences", y=0.98, fontsize=12)
plt.tight_layout(rect=[0, 0.08, 1, 0.95])

plt.savefig(PNG_PATH, dpi=1000, bbox_inches="tight", pad_inches=0.25)
plt.savefig(PDF_PATH, dpi=1000, bbox_inches="tight", pad_inches=0.25)
print(f"✅ Saved:\n- {PNG_PATH}\n- {PDF_PATH}")

plt.show()



In [ ]:
# -*- coding: utf-8 -*-
# Figure: Quadrant-level hazard composition by season (stacked bar %)
# pip install geopandas shapely matplotlib pandas pyproj fiona

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from pathlib import Path

# ===================== USER PATHS =====================
OUT_DIR = Path(r"D:\projects\climate\exports")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PNG_PATH = OUT_DIR / "khuzestan_stackedbar_FALL_SPRING.png"
#PDF_PATH = OUT_DIR / "khuzestan_stackedbar_FALL_SPRING.pdf"

SEASON_FILES = {
    "FALL": {
        "NW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedNW.csv",
        "NE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedNE.csv",
        "SW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedSW.csv",
        "SE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedSE.csv",
    },
    "SPRING": {
        "NW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedNW.csv",
        "NE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedNE.csv",
        "SW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedSW.csv",
        "SE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedSE.csv",
    }
}

# ===================== CONSTANTS =====================
HAZ_BIN = [
    "hazard_drought",
    "hazard_rain_deficit",
    "hazard_heatwave",
    "hazard_hot_and_dry",
    "hazard_cold_spell",
    "hazard_radiation_deficit",
    "hazard_temperature_accumulation_deficit",
]
HAZ_LABELS_EN = {
    "hazard_drought": "Drought",
    "hazard_rain_deficit": "Rain Deficit",
    "hazard_heatwave": "Heatwave",
    "hazard_hot_and_dry": "Hot & Dry (Compound)",
    "hazard_cold_spell": "Cold Spell",
    "hazard_radiation_deficit": "Radiation Deficit",
    "hazard_temperature_accumulation_deficit": "Temperature Accumulation Deficit",
}
HAZ_COLORS = {
    "hazard_drought": "#e41a1c",
    "hazard_rain_deficit": "#377eb8",
    "hazard_heatwave": "#ff7f00",
    "hazard_hot_and_dry": "#a65628",
    "hazard_cold_spell": "#984ea3",
    "hazard_radiation_deficit": "#dede00",
    "hazard_temperature_accumulation_deficit": "#4daf4a",
}

# اگر فقط *_prob داشته باشی حضور را با این آستانه تعریف می‌کنیم:
PROB_THRESHOLD = 0.5

# ===================== HELPERS =====================
def read_table_any(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")
    suf = p.suffix.lower()
    if suf in [".csv", ".txt"]:
        return pd.read_csv(p)
    if suf == ".parquet":
        return pd.read_parquet(p)
    if suf in [".xlsx", ".xls"]:
        return pd.read_excel(p)
    return pd.read_csv(p)

def compute_presence_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """
    خروجی: دیتافریمی با ستون‌های HAZ_BIN به صورت 0/1 (presence).
    اگر هر دو نوع ستون (binary و *_prob) موجود بود، presence = binary OR (prob>=thr)
    """
    out = pd.DataFrame(index=df.index)
    for base in HAZ_BIN:
        bin_col  = base if base in df.columns else None
        prob_col = base + "_prob" if base + "_prob" in df.columns else None
        bin_vals  = df[bin_col].astype(float) if bin_col else 0.0
        prob_vals = (df[prob_col] >= PROB_THRESHOLD).astype(float) if prob_col else 0.0
        out[base] = ((pd.Series(bin_vals, index=df.index).fillna(0) +
                      pd.Series(prob_vals, index=df.index).fillna(0)) > 0).astype(int)
    return out


def load_season(files_dict: dict) -> pd.DataFrame:
    """چهار فایل یک فصل را می‌خواند و ستون quad را از نام فایل‌ها می‌سازد."""
    frames = []
    for quad, p in files_dict.items():
        df = read_table_any(p)
        if "quad" not in df.columns:
            df = df.copy()
            df["quad"] = quad
        pres = compute_presence_matrix(df)
        for col in HAZ_BIN:
            if col not in df.columns:
                df[col] = pres[col]
            else:
                # اگر ستون دودویی هست ولی NaN دارد، با presence پر می‌کنیم
                df[col] = df[col].fillna(pres[col]).astype(int)
        frames.append(df[["quad", *HAZ_BIN]])
    return pd.concat(frames, ignore_index=True)

def build_quad_percent(df: pd.DataFrame) -> pd.DataFrame:
    """
    بازگشت: DataFrame درصد هازاردها برای هر ناحیه (سطر = quad، ستون = hazard)
    """
    counts = df.groupby("quad")[HAZ_BIN].sum()                         # occurrences
    pct = counts.div(counts.sum(axis=1), axis=0) * 100                 # to %
    # مرتب‌سازی ستون‌ها برای نمایش ثابت
    return pct.reindex(columns=HAZ_BIN)

# ===================== LOAD =====================
fall_df   = load_season(SEASON_FILES["FALL"])
spring_df = load_season(SEASON_FILES["SPRING"])

fall_pct   = build_quad_percent(fall_df)
spring_pct = build_quad_percent(spring_df)

# ===================== PLOT =====================
mpl.rcParams.update({
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "legend.fontsize": 8
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=450, sharey=True)
quad_order = ["NW", "NE", "SW", "SE"]
for ax, (season_name, pct_df) in zip(axes, [("Autumn", fall_pct), ("SPRING", spring_pct)]):
    # اطمینان از ترتیب نواحی
    pct_df = pct_df.reindex(quad_order)

    bottom = np.zeros(len(pct_df))
    for haz in HAZ_BIN:
        vals = pct_df[haz].to_numpy()
        ax.barh(quad_order, vals, left=bottom,
                color=HAZ_COLORS[haz], edgecolor="black", linewidth=0.3,
                label=HAZ_LABELS_EN[haz])
        bottom += vals

    ax.set_title(f"{season_name} — Quadrant hazard composition")
    ax.set_xlabel("Percentage (%)")
    ax.set_xlim(0, 100)
    ax.grid(axis="x", linestyle=":", alpha=0.35)

# Legend مشترک
legend_elems = [Line2D([0],[0], marker='s', color='w',
                       label=HAZ_LABELS_EN[k],
                       markerfacecolor=HAZ_COLORS[k], markeredgecolor="black",
                       markersize=8)
                for k in HAZ_BIN]
fig.legend(handles=legend_elems, loc="lower center", ncol=3,
           title="Hazard Types", fontsize=8, title_fontsize=9)

fig.suptitle("Khuzestan — Quadrant-level hazard composition by season (stacked %)", y=0.98, fontsize=11)
plt.tight_layout(rect=[0, 0.10, 1, 0.95])

plt.savefig(PNG_PATH, dpi=1000, bbox_inches="tight", pad_inches=0.25)
#plt.savefig(PDF_PATH, dpi=1000, bbox_inches="tight", pad_inches=0.25)
#print(f"✅ Saved:\n- {PNG_PATH}\n- {PDF_PATH}")

plt.show()


In [ ]:
# pip install geopandas shapely matplotlib pandas pyproj fiona
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from shapely.geometry import LineString, MultiLineString, Point, box
from shapely.ops import split
from pathlib import Path
import unicodedata
import random

# ===================== USER PATHS =====================
ADMIN1_PATH = r"D:\projects\climate\natural_earth_vector\10m_cultural\ne_10m_admin_1_states_provinces.shp"

# هر فصل چهار فایل (NW/NE/SW/SE). پسوند CSV/Parquet/Excel فرقی ندارد.
SEASON_FILES = {
    "FALL": {
        "NW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedNW.csv",
        "NE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedNE.csv",
        "SW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedSW.csv",
        "SE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\FALL_cleanedSE.csv",
    },
    "SPRING": {
        "NW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedNW.csv",
        "NE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedNE.csv",
        "SW": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedSW.csv",
        "SE": r"D:\projects\climate\climate\DATA\raw-data\seasonal-merged-data\MODIFIED\SPRING_cleanedSE.csv",
    }
}

OUT_DIR = Path(r"D:\projects\climate\exports")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PNG_PATH = OUT_DIR / "khuzestan_side_by_side_FALL_SPRING.png"
PDF_PATH = OUT_DIR / "khuzestan_side_by_side_FALL_SPRING.pdf"

# ===================== CONSTANTS =====================
# study bbox (Khuzestan)
LAT_MIN, LAT_MAX = 29.5, 33.0
LON_MIN, LON_MAX = 47.0, 51.0

# internal split lines for quadrants
LON_SPLIT = 49.0
LAT_SPLIT = 31.25

# hazards
HAZ_BIN = [
    "hazard_drought",
    "hazard_rain_deficit",
    "hazard_heatwave",
    "hazard_hot_and_dry",
    "hazard_cold_spell",
    "hazard_radiation_deficit",
    "hazard_temperature_accumulation_deficit",
]
HAZ_PROB = [c + "_prob" for c in HAZ_BIN]

HAZ_LABELS_EN = {
    "hazard_drought": "Drought",
    "hazard_rain_deficit": "Rain Deficit",
    "hazard_heatwave": "Heatwave",
    "hazard_hot_and_dry": "Hot & Dry",
    "hazard_cold_spell": "Cold Spell",
    "hazard_radiation_deficit": "Radiation",
    "hazard_temperature_accumulation_deficit": "Temp Acc",
}

# ثابت و خوانا برای مقاله
# رنگ‌های پرکنتراست (Tableau/ColorBrewer ترکیبی)
HAZ_COLORS = {
    "hazard_drought": "#e41a1c",        # red
    "hazard_rain_deficit": "#377eb8",   # blue
    "hazard_heatwave": "#ff7f00",       # orange
    "hazard_hot_and_dry": "#a65628",    # brown
    "hazard_cold_spell": "#984ea3",     # purple
    "hazard_radiation_deficit": "#dede00", # yellow
    "hazard_temperature_accumulation_deficit": "#4daf4a",  # green
}

# شکل مارکر برای هر هازارد (برای چند-هازارد شدن هر نقطه)
HAZ_MARKERS = {
    "hazard_drought": "o",        # circle
    "hazard_rain_deficit": "s",   # square
    "hazard_heatwave": "^",       # triangle_up
    "hazard_hot_and_dry": "P",    # plus-filled
    "hazard_cold_spell": "D",     # diamond
    "hazard_radiation_deficit": "*", # star
    "hazard_temperature_accumulation_deficit": "X", # x-filled
}


# اگر فقط *_prob داشتی، آستانه‌ی فعال بودن رو اینجا تعیین کن:
PROB_THRESHOLD = 0.5

# ===================== HELPERS =====================
def norm(s: str) -> str:
    return unicodedata.normalize("NFKD", str(s)).encode("ascii","ignore").decode("ascii").lower()

def pick_name_col(gdf):
    for cand in ["name","name_en","name_local","gn_name","prov_name","NAME_1","name_1"]:
        if cand in gdf.columns: return cand
    raise ValueError(f"No province name column found. Columns: {list(gdf.columns)}")


def read_table_any(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")
    suf = p.suffix.lower()
    if suf in [".csv", ".txt"]:
        return pd.read_csv(p)
    if suf in [".parquet"]:
        return pd.read_parquet(p)
    if suf in [".feather"]:
        import pyarrow.feather as ft
        return ft.read_feather(p)
    if suf in [".xlsx", ".xls"]:
        return pd.read_excel(p)
    return pd.read_csv(p)

def assign_quadrant(lat, lon):
    if lat >= LAT_SPLIT and lon <  LON_SPLIT: return "NW"
    if lat >= LAT_SPLIT and lon >= LON_SPLIT: return "NE"
    if lat <  LAT_SPLIT and lon <  LON_SPLIT: return "SW"
    return "SE"

def compute_presence_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """
    خروجی: DataFrame فقط با ستون‌های HAZ_BIN که 0/1 هستند (presence).
    - اگر *_prob هست: presence = (prob >= PROB_THRESHOLD)
    - اگر hazard_* (0/1) هست: همان را استفاده می‌کنیم.
    - اگر هر دو باشند: presence = (hazard_*)  |  (prob>=thr)
    """
    out = pd.DataFrame(index=df.index)
    for base in HAZ_BIN:
        bin_col = base if base in df.columns else None
        prob_col = base + "_prob" if base + "_prob" in df.columns else None

        bin_vals = df[bin_col].astype(float) if bin_col else 0.0
        prob_vals = (df[prob_col] >= PROB_THRESHOLD).astype(float) if prob_col else 0.0

        out[base] = (pd.Series(bin_vals, index=df.index).fillna(0) + pd.Series(prob_vals, index=df.index).fillna(0) > 0).astype(int)
    return out

def load_season_data(files_dict: dict) -> pd.DataFrame:
    """ چهارفایل فصل را بخوان، به باکس فیلتر کن، presence بساز، و quad تخصیص بده. """
    dfs = []
    for q, path in files_dict.items():
        df = read_table_any(path)
        if ("latitude" not in df.columns) or ("longitude" not in df.columns):
            raise KeyError(f"latitude/longitude missing in: {path}")
        # bbox filter
        df = df[(df["latitude"].between(LAT_MIN, LAT_MAX)) & (df["longitude"].between(LON_MIN, LON_MAX))]
        if df.empty:
            print(f"Warning: no rows in bbox for {q}: {path}")
        # presence matrix
        pres = compute_presence_matrix(df)
        # attach back
        df = df.copy()
        df[HAZ_BIN] = pres
        # quad
        df["quad"] = [assign_quadrant(lat, lon) for lat, lon in zip(df["latitude"], df["longitude"])]
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True, sort=False)
def top2_hazards_per_point(df: pd.DataFrame) -> pd.DataFrame:
    """
    برای هر نقطه (lat, lon) تعداد حضور هر hazard را جمع می‌زند
    و دو تای پرتکرار را برمی‌گرداند.
    خروجی: DataFrame با ستون‌های:
      latitude, longitude, quad, top1, top2
    """
    # شمارش حضور به ازای هر نقطه
    agg = (df.groupby(["latitude", "longitude", "quad"])[HAZ_BIN]
             .sum()
             .reset_index())

    records = []
    for _, row in agg.iterrows():
        counts = row[HAZ_BIN]
        # اگر هیچ هازاردی نباشد، رد کن
        if counts.sum() == 0:
            continue
        top = counts.sort_values(ascending=False).index.tolist()
        top1 = top[0]
        top2 = top[1] if len(top) > 1 and counts.iloc[1] > 0 else None
        records.append({
            "latitude":  row["latitude"],
            "longitude": row["longitude"],
            "quad":      row["quad"],
            "top1":      top1,
            "top2":      top2
        })
    return pd.DataFrame(records)


# ===================== GEOMETRY: KHUZESTAN + QUADRANTS =====================
assert Path(ADMIN1_PATH).exists(), f"Admin-1 path not found: {ADMIN1_PATH}"
adm1 = gpd.read_file(ADMIN1_PATH).to_crs(epsg=4326)
name_col = pick_name_col(adm1)

country_cols = [c for c in adm1.columns if c.lower() in ["adm0_a3","iso_a3","admin","geounit","sr_adm0_a3"]]
if country_cols:
    ccol = country_cols[0]
    mask_ir = adm1[ccol].astype(str).str.contains("IR|IRN|Iran", case=False, na=False)
    adm1 = adm1[mask_ir].copy() if mask_ir.any() else adm1


alt_names = ["khuzestan","khuzistan","khuzestan province","خوزستان"]
khz = adm1[adm1[name_col].apply(lambda x: any(a in norm(x) for a in alt_names))]
if khz.empty:
    raise ValueError("Khuzestan not found in Admin-1 layer.")
khz_geom = khz.geometry.union_all() if hasattr(khz.geometry, "union_all") else khz.geometry.unary_union

# clip to bbox
bbox_poly = box(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)
khz_geom = khz_geom.intersection(bbox_poly)

# split into 4 quadrants
vline = LineString([(LON_SPLIT, LAT_MIN-1), (LON_SPLIT, LAT_MAX+1)])
hline = LineString([(LON_MIN-1, LAT_SPLIT), (LON_MAX+1, LAT_SPLIT)])
pieces = split(khz_geom, MultiLineString([vline, hline]))
labels, geoms = [], []
for g in pieces.geoms:
    c = g.representative_point()
    if c.y >= LAT_SPLIT and c.x <  LON_SPLIT: lab = "NW"
    elif c.y >= LAT_SPLIT and c.x >= LON_SPLIT: lab = "NE"
    elif c.y <  LAT_SPLIT and c.x <  LON_SPLIT: lab = "SW"
    else: lab = "SE"
    labels.append(lab); geoms.append(g)
quad_gdf = gpd.GeoDataFrame({"quad": labels}, geometry=geoms, crs="EPSG:4326")

# ===================== LOAD DATA (FALL + SPRING) =====================

fall_df   = load_season_data(SEASON_FILES["FALL"])
spring_df = load_season_data(SEASON_FILES["SPRING"])

fall_top2_points   = top2_hazards_per_point(fall_df)
spring_top2_points = top2_hazards_per_point(spring_df)


# برای بارچارت‌ها: درصد هازاردها در کل محدوده (presence-based, multi-hazard)
def season_percentages(df: pd.DataFrame) -> pd.Series:
    # شمارش وقوع‌ها (هر حضور 1 حساب می‌شود)
    counts = df[HAZ_BIN].sum().astype(int)
    total = counts.sum()
    return (counts / total * 100).sort_values(ascending=True) if total > 0 else counts

fall_pct   = season_percentages(fall_df)
spring_pct = season_percentages(spring_df)

# برای لیبل نواحی: درصد هر هازارد در هر ناحیه و انتخاب Top-2
def top2_per_quad(df: pd.DataFrame):
    by_quad = df.groupby("quad")[HAZ_BIN].sum()
    # درصد در هر ناحیه (normalize across columns)
    pct_by_quad = by_quad.div(by_quad.sum(axis=1), axis=0) * 100
    out = {}
    for q, row in pct_by_quad.iterrows():
        top2 = row.sort_values(ascending=False).head(2)
        out[q] = top2  # Series
    return out, pct_by_quad  # برای نیاز بعدی اگر خواستی

fall_top2, fall_pct_by_quad     = top2_per_quad(fall_df)
spring_top2, spring_pct_by_quad = top2_per_quad(spring_df)

# ===================== PLOTTING =====================
# ===================== PLOTTING =====================
mpl.rcParams.update({
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "legend.fontsize": 7
})

fig = plt.figure(figsize=(14, 10), dpi=400)
gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1.2, 0.8], wspace=0.15, hspace=0.25)

# ------ MAP helper: draw season map in (row,col) ------
def draw_season_map(ax, season_name: str, top2_df: pd.DataFrame):
    # زمینه: خوزستان + مرز نواحی
    gpd.GeoSeries([khz_geom], crs="EPSG:4326").plot(ax=ax, facecolor="#f2f2f2",
                                                    edgecolor="#aaaaaa", linewidth=0.6)
    quad_gdf.plot(ax=ax, facecolor="none", edgecolor="#111111", linewidth=0.9)
    gpd.GeoSeries([vline, hline], crs="EPSG:4326").plot(ax=ax, color="black", linewidth=0.7, alpha=0.6)

    # فقط دو هازارد برتر هر نقطه (offset کوچک برای تفکیک دو مارکر)
    dx, dy = 0.03, 0.0
    if not top2_df.empty:
        # top1
        t1 = top2_df.dropna(subset=["top1"]).copy()
        for h, sub in t1.groupby("top1"):
            ax.scatter(sub["longitude"]-dx, sub["latitude"]+dy,
                       s=20, alpha=0.9, linewidths=0.5, edgecolors="black",
                       c=HAZ_COLORS[h], marker=HAZ_MARKERS[h])
        # top2
        t2 = top2_df.dropna(subset=["top2"]).copy()
        for h, sub in t2.groupby("top2"):
            ax.scatter(sub["longitude"]+dx, sub["latitude"]-dy,
                       s=14, alpha=0.85, linewidths=0.5, edgecolors="black",
                       c=HAZ_COLORS[h], marker=HAZ_MARKERS[h])


    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{season_name} — Top-2 hazards per grid point")

# ------ BAR helper: draw horizontal bar for a season ------
xmax = 1.1 * max(fall_pct.max(), spring_pct.max())  # محور x مشترک

def draw_season_bar(ax, season_name: str, pct_series: pd.Series, x_max: float):
    labels = [HAZ_LABELS_EN[h] for h in pct_series.index]
    colors = [HAZ_COLORS[h] for h in pct_series.index]

    y = np.arange(len(pct_series))
    bars = ax.barh(y, pct_series.values, color=colors, height=0.50)
    ax.set_yticks(y, labels)
    ax.set_xlabel("Percentage of occurrences (%)")
    ax.set_ylabel("")
    ax.set_title(f"{season_name} — Hazard composition (multi-hazard)")
    ax.grid(axis="x", linestyle=":", alpha=0.35)
    ax.set_xlim(0, x_max)
    ax.margins(y=0.22)
    ax.bar_label(bars, labels=[f"{v:.1f}%" for v in pct_series.values], padding=2, fontsize=7)

# --- Left column: FALL ---
ax_map_fall = fig.add_subplot(gs[0, 0])
draw_season_map(ax_map_fall, "Autumn", fall_top2_points)

ax_bar_fall = fig.add_subplot(gs[1, 0])
draw_season_bar(ax_bar_fall, "Autumn", fall_pct, xmax)

# --- Right column: SPRING ---
ax_map_spr = fig.add_subplot(gs[0, 1])
draw_season_map(ax_map_spr, "SPRING", spring_top2_points)  # ✅ اینجا top2 ها را پاس بده

ax_bar_spr = fig.add_subplot(gs[1, 1])
draw_season_bar(ax_bar_spr, "SPRING", spring_pct, xmax)

# Legend مشترک پایین شکل
legend_elems = [
    Line2D([0],[0], marker=HAZ_MARKERS["hazard_drought"], color='w',
           label="Drought", markerfacecolor=HAZ_COLORS["hazard_drought"], markeredgecolor="black", markersize=7),
    Line2D([0],[0], marker=HAZ_MARKERS["hazard_rain_deficit"], color='w',
           label="Rain Deficit", markerfacecolor=HAZ_COLORS["hazard_rain_deficit"], markeredgecolor="black", markersize=7),
    Line2D([0],[0], marker=HAZ_MARKERS["hazard_heatwave"], color='w',
           label="Heatwave", markerfacecolor=HAZ_COLORS["hazard_heatwave"], markeredgecolor="black", markersize=7),
    Line2D([0],[0], marker=HAZ_MARKERS["hazard_hot_and_dry"], color='w',
           label="Hot & Dry (Compound)", markerfacecolor=HAZ_COLORS["hazard_hot_and_dry"], markeredgecolor="black", markersize=7),
    Line2D([0],[0], marker=HAZ_MARKERS["hazard_cold_spell"], color='w',
           label="Cold Spell", markerfacecolor=HAZ_COLORS["hazard_cold_spell"], markeredgecolor="black", markersize=7),
    Line2D([0],[0], marker=HAZ_MARKERS["hazard_radiation_deficit"], color='w',
           label="Radiation Deficit", markerfacecolor=HAZ_COLORS["hazard_radiation_deficit"], markeredgecolor="black", markersize=7),
    Line2D([0],[0], marker=HAZ_MARKERS["hazard_temperature_accumulation_deficit"], color='w',
           label="Temperature Accumulation Deficit", markerfacecolor=HAZ_COLORS["hazard_temperature_accumulation_deficit"], markeredgecolor="black", markersize=7),
]

fig.legend(handles=legend_elems, loc="lower center", ncol=3,
           title="Hazard Types", fontsize=8, title_fontsize=9)



plt.tight_layout(rect=[0, 0.06, 1, 0.95])

# SAVE (high-res) then show
plt.savefig(PNG_PATH, dpi=1000, bbox_inches="tight", pad_inches=0.25)
plt.savefig(PDF_PATH, dpi=1000, bbox_inches="tight", pad_inches=0.25)
print(f"✅ Saved:\n- {PNG_PATH}\n- {PDF_PATH}")

plt.show()
